# 🧼 Clean & Visualize VCT Agent Composition Data (2021–2025)
This notebook loads yearly `teams_picked_agents.csv` files, cleans them, splits agents into columns, and generates visual summaries.

## 📦 Step 1: Import Required Libraries

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
sns.set(style='whitegrid')

## 📁 Step 2: Set Up Paths and Output Folder

In [ ]:
# Use the project-relative source data folder
root_path = os.path.join("kaggle", "input", "valorant-champion-tour-2021-2023-data")
output_dir = 'cleaned_data'
os.makedirs(output_dir, exist_ok=True)
merged_all_years = []

## 🧹 Step 3: Clean, Split Agents, and Save Data by Year

In [ ]:
for folder_name in os.listdir(root_path):
    if folder_name.startswith('vct_') and folder_name[4:].isdigit():
        year = folder_name[-4:]
        folder_path = os.path.join(root_path, folder_name)
        file_path = os.path.join(folder_path, 'agents', 'teams_picked_agents.csv')

        if not os.path.exists(file_path):
            print(f"Skipping missing file: {file_path}")
            continue

        data = pd.read_csv(file_path)
        data['Win Rate'] = data['Total Wins By Map'] / data['Total Maps Played']
        data['Win Rate'] = data['Win Rate'].fillna(0)
        data['Win'] = (data['Total Wins By Map'] > data['Total Loss By Map']).astype(int)

        cleaned_data = (
            data.groupby(['Tournament', 'Stage', 'Match Type', 'Map', 'Team'])
            .agg({
                'Agent': lambda x: ', '.join(sorted(x)),
                'Win': 'max'
            })
            .reset_index()
        )

        cleaned_data.rename(columns={'Agent': 'Agents Played', 'Win': 'Match Outcome'}, inplace=True)
        cleaned_data['Year'] = int(year)

        # Split agents into separate columns
        agent_split = cleaned_data['Agents Played'].str.split(', ', expand=True)
        for i in range(5):
            cleaned_data[f'Agent_{i+1}'] = agent_split[i] if i in agent_split.columns else None

        merged_all_years.append(cleaned_data)
        output_file = os.path.join(output_dir, f'cleaned_match_data_{year}.csv')
        cleaned_data.to_csv(output_file, index=False)
        print(f"✅ Saved: {output_file}")

## 📦 Step 4: Combine All Years into Master File

In [ ]:
if merged_all_years:
    all_years_df = pd.concat(merged_all_years, ignore_index=True)
    master_path = os.path.join(output_dir, 'cleaned_match_data_all_years.csv')
    all_years_df.to_csv(master_path, index=False)
    print(f"🎉 Combined file saved to: {master_path}")
else:
    print("⚠️ No data was processed.")

## 📊 Step 5: Visual Summary — Matches per Year

In [ ]:
if not all_years_df.empty:
    plt.figure(figsize=(8, 5))
    sns.countplot(data=all_years_df, x='Year', palette='Blues_d')
    plt.title('Number of Matches per Year')
    plt.xlabel('Year')
    plt.ylabel('Match Count')
    plt.tight_layout()
    plt.show()

## 📊 Step 6: Agent Usage per Map per Year

In [ ]:
# Agent usage per map in a given year
target_year = 2023  # Change this value for different years
map_agent = all_years_df[all_years_df['Year'] == target_year][['Map', 'Agent_1', 'Agent_2', 'Agent_3', 'Agent_4', 'Agent_5']]
map_agent_long = map_agent.melt(id_vars='Map', value_name='Agent').dropna()
agent_map_summary = map_agent_long.groupby(['Map', 'Agent']).size().reset_index(name='Pick Count')

plt.figure(figsize=(12, 7))
sns.barplot(data=agent_map_summary, x='Pick Count', y='Agent', hue='Map')
plt.title(f'Agent Picks per Map in {target_year}')
plt.tight_layout()
plt.show()